# ShiftLog-Gym Smoke Test

Sanity-check the environment, compare simple baselines, and export first-pass memory-policy metrics.

In [ ]:
# Colab setup: clone repo + install it so `shiftlog_gym` is importable.
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip -q install -e .


In [ ]:
!pip -q install pandas matplotlib seaborn transformers accelerate


In [ ]:
import ast
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from shiftlog_gym.simulator import ShiftLogSimulator
from shiftlog_gym.training import (
    TEST_VARIANTS,
    AVAILABLE_TOOLS,
    build_variant_split,
    collect_policy_rollouts,
    random_policy,
    rollout_prompted_policy,
    scripted_policy,
    summarize_episode,
    write_artifacts,
)

sns.set_theme(style='whitegrid')


In [ ]:
split_config = build_variant_split()
pd.DataFrame([(name, list(indices)) for name, indices in split_config.items()], columns=['split', 'variant_indices'])


In [ ]:
sim = ShiftLogSimulator()
obs = sim.reset(family='db_pool_exhaustion', variant_index=0)
print(obs)


In [ ]:
random_rows, random_memory, random_tools = collect_policy_rollouts(
    policy_name='random',
    policy_fn=random_policy,
    split='test',
    variants=TEST_VARIANTS,
    seeds=(0,),
)

scripted_rows, scripted_memory, scripted_tools = collect_policy_rollouts(
    policy_name='scripted',
    policy_fn=scripted_policy,
    split='test',
    variants=TEST_VARIANTS,
    seeds=(0,),
)

baseline_rows = random_rows + scripted_rows
baseline_df = pd.DataFrame(baseline_rows).sort_values(['episode_name']).reset_index(drop=True)
baseline_df[['episode_name', 'family', 'weighted_reward', 'recall_before_action_rate', 'linked_incident_success_rate', 'memory_precision', 'contradiction_rate', 'unsupported_mitigation_rate']]


In [ ]:
artifact_dir = Path('artifacts/smoke')
write_artifacts(artifact_dir / 'random', random_rows, random_memory, random_tools)
write_artifacts(artifact_dir / 'scripted', scripted_rows, scripted_memory, scripted_tools)
write_artifacts(artifact_dir / 'combined', baseline_rows, random_memory + scripted_memory, random_tools + scripted_tools)
print('Wrote smoke-test artifacts to', artifact_dir)


In [ ]:
plot_columns = [
    'weighted_reward',
    'recall_before_action_rate',
    'linked_incident_success_rate',
    'memory_precision',
    'contradiction_rate',
]

fig, axes = plt.subplots(1, len(plot_columns), figsize=(18, 4))
for axis, column in zip(axes, plot_columns):
    sns.barplot(data=baseline_df, x='episode_name', y=column, ax=axis)
    axis.set_title(column)
    axis.tick_params(axis='x', rotation=90)
fig.tight_layout()
plt.show()


## Optional prompted-base baseline

This is intentionally off by default on free Colab because it adds inference overhead. Turn it on if you want a raw-model baseline before the RL notebook.

In [ ]:
RUN_PROMPTED_BASELINE = False
PROMPTED_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
PROMPTED_MAX_NEW_TOKENS = 180


In [ ]:
if RUN_PROMPTED_BASELINE:
    import re
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(PROMPTED_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        PROMPTED_MODEL_NAME,
        torch_dtype='auto',
        device_map='auto',
    )

    def extract_action_dict(text: str):
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return {
                'tool': 'inspect_service',
                'arguments': {'service': 'payments-api'},
            }
        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            parsed = ast.literal_eval(match.group(0))
        if parsed.get('tool') not in AVAILABLE_TOOLS:
            return {
                'tool': 'inspect_service',
                'arguments': {'service': 'payments-api'},
            }
        return parsed

    def prompted_policy(observation, transcript):
        prompt = f"""
You are an SRE agent inside ShiftLog-Gym.
Pick exactly one tool call in JSON.
Use this schema only: {{"tool": str, "arguments": dict}}.
Prefer read_shift_log before state-changing actions on repeated incidents.
Available tools: {', '.join(AVAILABLE_TOOLS)}.
Current observation:\n{observation}\n
Recent transcript:\n{json.dumps(transcript[-2:], indent=2)}
"""
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=PROMPTED_MAX_NEW_TOKENS, do_sample=False)
        decoded = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return extract_action_dict(decoded)

    prompted_rows = []
    prompted_memory = []
    prompted_tools = []
    for family in ['db_pool_exhaustion', 'auth_timeout_cascade', 'memory_oom_signature', 'feature_flag_regression']:
        simulator = ShiftLogSimulator()
        simulator.reset(seed=0, family=family, variant_index=7)
        rollout_prompted_policy(simulator, prompted_policy, max_steps=18)
        artifacts = summarize_episode(simulator, f'prompted-{family}', 'test', 0, 7)
        prompted_rows.append(artifacts.episode_row)
        prompted_memory.extend(artifacts.memory_events)
        prompted_tools.extend(artifacts.tool_timeline)
    prompted_df = pd.DataFrame(prompted_rows)
    write_artifacts(artifact_dir / 'prompted_base', prompted_rows, prompted_memory, prompted_tools)
    display(prompted_df)
else:
    print('Prompted baseline skipped. Set RUN_PROMPTED_BASELINE = True to enable it.')
